#### Importing the required libraries

In [29]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MinMaxScaler

#### Extracting the cached striker data

In [37]:
# Reading the csv file for the striker data and the originsl dataset as well

df = pd.read_csv('../data/wc_2022_events.csv', low_memory=False)
df_clean = df.dropna(subset=['player']).copy()
df_clean['player'] = df_clean['player'].astype(str).str.strip()

df_scouted_strikers = pd.read_csv('../data/wc_2022_striker_report.csv', index_col='player')
df_scouted_strikers.index = df_scouted_strikers.index.astype(str).str.strip()
print(df_scouted_strikers.head())

                   Goals  Total_xG  Avg_xG_per_Shot  Shot_Count  G_minus_xG  \
player                                                                        
Gue-Sung Cho           2  0.712589         0.089074           8    1.287411   
Olivier Giroud         4  3.035955         0.178586          17    0.964045   
Youssef En-Nesyri      2  1.045874         0.095079          11    0.954126   
Julián Álvarez         4  1.908327         0.173484          11    2.091673   
Bruno Petković         1  0.099778         0.099778           1    0.900222   

                   Aerial_Duels_Won  Aerial_Duels_Total  Aerial_Pass_Wins  \
player                                                                      
Gue-Sung Cho                      0                   0                16   
Olivier Giroud                    0                   0                18   
Youssef En-Nesyri                 0                   0                15   
Julián Álvarez                    3                   0      

#### Normalizing data per 90 minutes played

In [33]:
# Identify the total minutes played by each player

minutes_per_match = df_clean.groupby(['player', 'match_id'])['minute'].max().reset_index()
player_minutes = minutes_per_match.groupby('player')['minute'].sum().rename('Total_Minutes')

prep_df = df_scouted_strikers.reset_index().merge(
    player_minutes.reset_index(), 
    on='player', 
    how='left'
)

prep_df = prep_df[prep_df['Total_Minutes'] >= 180].copy()
prep_df.set_index('player', inplace=True)

print(prep_df.head())

                   Goals  Total_xG  Avg_xG_per_Shot  Shot_Count  G_minus_xG  \
player                                                                        
Gue-Sung Cho           2  0.712589         0.089074           8    1.287411   
Olivier Giroud         4  3.035955         0.178586          17    0.964045   
Youssef En-Nesyri      2  1.045874         0.095079          11    0.954126   
Julián Álvarez         4  1.908327         0.173484          11    2.091673   
Bruno Petković         1  0.099778         0.099778           1    0.900222   

                   Aerial_Duels_Won  Aerial_Duels_Total  Aerial_Pass_Wins  \
player                                                                      
Gue-Sung Cho                      0                   0                16   
Olivier Giroud                    0                   0                18   
Youssef En-Nesyri                 0                   0                15   
Julián Álvarez                    3                   0      

In [35]:
# Identify the metrics to normalize and normalize them

metrics_to_normalize = ['Goals', 'Total_xG', 'Total_Aerial_Wins']
for metric in metrics_to_normalize:
    prep_df[f'{metric}_p90'] = (prep_df[metric] / prep_df['Total_Minutes']) * 90

print(prep_df.head())

                   Goals  Total_xG  Avg_xG_per_Shot  Shot_Count  G_minus_xG  \
player                                                                        
Gue-Sung Cho           2  0.712589         0.089074           8    1.287411   
Olivier Giroud         4  3.035955         0.178586          17    0.964045   
Youssef En-Nesyri      2  1.045874         0.095079          11    0.954126   
Julián Álvarez         4  1.908327         0.173484          11    2.091673   
Bruno Petković         1  0.099778         0.099778           1    0.900222   

                   Aerial_Duels_Won  Aerial_Duels_Total  Aerial_Pass_Wins  \
player                                                                      
Gue-Sung Cho                      0                   0                16   
Olivier Giroud                    0                   0                18   
Youssef En-Nesyri                 0                   0                15   
Julián Álvarez                    3                   0      

In [36]:
# Scaling the data using min-max scaler

scaler = MinMaxScaler()

cols_to_scale = [f'{m}_p90' for m in metrics_to_normalize] + ['G_minus_xG']
prep_df[[f'scaled_{c}' for c in cols_to_scale]] = scaler.fit_transform(prep_df[cols_to_scale])

print(f"Pre-processing complete. {len(prep_df)} candidates remain after minute-filtering.")
print(prep_df[[col for col in prep_df.columns if 'p90' in col]].head())

Pre-processing complete. 68 candidates remain after minute-filtering.
                   Goals_p90  Total_xG_p90  Total_Aerial_Wins_p90  \
player                                                              
Gue-Sung Cho        0.493151      0.175707               3.945205   
Olivier Giroud      0.750000      0.569242               3.375000   
Youssef En-Nesyri   0.327869      0.171455               2.459016   
Julián Álvarez      0.599002      0.285773               0.898502   
Bruno Petković      0.161002      0.016064               2.254025   

                   scaled_Goals_p90  scaled_Total_xG_p90  \
player                                                     
Gue-Sung Cho               0.421918             0.179955   
Olivier Giroud             0.641667             0.597545   
Youssef En-Nesyri          0.280510             0.175443   
Julián Álvarez             0.512479             0.296749   
Bruno Petković             0.137746             0.010554   

                   scaled

### Creating Binning based on Tactical Profiles

In [38]:
# Create Tactical Archetype Flags
prep_df['is_aerial_specialist'] = (prep_df['scaled_Total_Aerial_Wins_p90'] > 0.8).astype(int)
prep_df['is_clinical_finisher'] = (prep_df['scaled_G_minus_xG'] > 0.7).astype(int)
prep_df['is_high_volume_threat'] = (prep_df['scaled_Total_xG_p90'] > 0.7).astype(int)

#### Caching pre-processed data for modelling

In [40]:
output_path = '../data/wc_2022_pre_processed_striker_data.csv'
prep_df.to_csv(output_path)
print(f"Pre-processing complete. {len(prep_df)} players data exported")

Pre-processing complete. 68 players data exported
